# L16 · 翻转矩阵 — 行列式与逆

> 备注：本节后半部分保留一组外部线性代数课件的行列式与逆矩阵例题。

**目标**：会算 2×2 行列式与逆；理解 3×3 行列式的**余子式展开**、**伴随矩阵**求逆 `A⁻¹ = adj(A)/|A|`。

**为什么对 Aurora 重要**：在 Aurora 特征提取中，`det(M) = 0` 意味着协方差矩阵奇异、白化步骤无法进行；`A⁻¹` 直接出现在 Mahalanobis 距离计算和最小二乘求解里。

## 本课剧情：检查空间有没有被压扁

行列式衡量矩阵对体积的缩放比例：det=2 表示面积翻倍，det=0 表示空间被压扁、信息丢失、矩阵不可逆。逆矩阵是变换的撤销操作，只有 det≠0 时才存在。这节课用 2×2 和 3×3 两个具体例子把这两件事算清楚。

## 1. 2×2：`det = ad − bc`，逆有闭式公式

$$A=\begin{pmatrix}a&b\\c&d\end{pmatrix},\quad A^{-1}=\frac{1}{ad-bc}\begin{pmatrix}d&-b\\-c&a\end{pmatrix}$$

`ad − bc` 的几何意义是矩阵两列张成的平行四边形有向面积：当两行（或两列）成比例时，平行四边形退化为线段，面积为零，整个平面被压缩成一维，信息不可还原。在数值计算中，det 极小（如 1e-15）即使不等于零也会在求逆时放大舍入误差——此时用 `np.linalg.cond(A)` 检查条件数更实用：条件数超过 1e12 意味着逆矩阵的精度已不可信。

## 符号入口：先看形状，再看运算

本节对象是方阵 `M`（shape `(n, n)`）。2×2 的行列式是 `ad−bc`，可以直接计算；3×3 的行列式通过余子式展开，把问题拆成三个 2×2 子式来处理。

## 2. ✏️ 实现 `det_2x2(M)` 与 `inv_2x2(M)`

**推理路线**：
1. 解包矩阵：`a, b = M[0]`，`c, d = M[1]`，得到四个标量
2. 计算行列式：`det = a*d - b*c`；这是两列向量 `[a,c]` 和 `[b,d]` 张成平行四边形的有向面积
3. 判断可逆性：`det == 0` 时矩阵把平面压成直线，无法求逆，应返回 `None` 而非除以零
4. 构造逆矩阵：对角元素互换（`d` 移到左上角，`a` 移到右下角），副对角元素取反，整体乘以 `1/det`

**参考输入输出**：`M=[[1,0],[0,2]]` → `det=2`，`inv=[[1,0],[0,0.5]]`；`M=[[1,2],[2,4]]` → `det=0`，不可逆

<details><summary>点击查看参考实现</summary>

```python
def det_2x2(M):
    a, b = M[0]
    c, d = M[1]
    return a*d - b*c

def inv_2x2(M):
    a, b = M[0]
    c, d = M[1]
    det = a*d - b*c
    if det == 0:
        return None
    return (1/det) * np.array([[d, -b], [-c, a]])
```

</details>

### 写代码前，先把变量表补完整

写 `det_2x2` 前明确三件事：
- 输入：`M`，shape `(2, 2)`，元素解包为 `a, b, c, d`
- 关键步骤：`det = a*d - b*c`；det 为 0 时无法求逆
- 返回：`det_2x2` 返回标量；`inv_2x2` 返回 shape `(2, 2)` 的矩阵

In [ ]:
import numpy as np
def det_2x2(M):
    # ✏️ TODO: 返回 ad - bc
    return None

def inv_2x2(M):
    # ✏️ TODO: 返回 2x2 逆矩阵
    return None


## 动手观察：det 值与可逆性

运行下面这格，观察 `det` 随矩阵系数的变化：当两行成比例时 det 变为 0，`inv_2x2` 应当拒绝求逆。

In [ ]:
import numpy as np

v = np.array([3.0, 4.0])
A = np.array([[2.0, 0.0],
              [0.0, 0.5]])

print('v =', v, 'shape =', v.shape)
print('A =')
print(A)
print('A shape =', A.shape)
print('A @ v =', A @ v)
print('向量长度 ||v|| =', np.linalg.norm(v))


## 代码实验：不同矩阵的 det 和逆

下面遍历几组矩阵，对比 det 值与逆矩阵是否存在。

In [ ]:
import numpy as np

A = np.array([[2.0, 1.0],
              [0.0, 1.0]])
vectors = [np.array([1.0, 0.0]), np.array([0.0, 1.0]), np.array([1.0, 1.0]), np.array([2.0, -1.0])]

print('A =')
print(A)
for v in vectors:
    out = A @ v
    print(f'v={v} -> A@v={out}')


In [ ]:
M = np.array([[4.,3.],[6.,3.]])
print('det =', det_2x2(M), ' (numpy:', round(np.linalg.det(M),6), ')')
assert abs(det_2x2(M) - np.linalg.det(M)) < 1e-9
assert np.allclose(inv_2x2(M), np.linalg.inv(M))
print('✅ 2×2 行列式与逆 通过。')

## 3. 3×3 行列式：按第一行**余子式展开**（补充例题）

补充例题：$A=\begin{pmatrix}1&1&0\\1&2&1\\0&1&3\end{pmatrix}$，答案 $|A|=2$。

$|A| = a_{11}M_{11} - a_{12}M_{12} + a_{13}M_{13}$，其中 $M_{ij}$ 是划掉第 i 行第 j 列后的 2×2 行列式(子式/minor)。


In [ ]:
A = np.array([[1,1,0],[1,2,1],[0,1,3]], float)
# 按第一行展开
M11 = np.linalg.det(A[1:,[1,2]])   # 划掉第0行第0列
M12 = np.linalg.det(A[1:,[0,2]])   # 划掉第0行第1列
M13 = np.linalg.det(A[1:,[0,1]])   # 划掉第0行第2列
detA = A[0,0]*M11 - A[0,1]*M12 + A[0,2]*M13
print('按余子式展开 |A| =', round(detA), ' (课件 2)')

## 4. 伴随矩阵求逆：`A⁻¹ = adj(A) / |A|`（补充例题）

补充例题结果（A 对称）：
$$A^{-1}=\tfrac12\begin{pmatrix}5&-3&1\\-3&3&-1\\1&-1&1\end{pmatrix}$$


In [ ]:
inv_expected = 0.5*np.array([[5,-3,1],[-3,3,-1],[1,-1,1]], float)
print('numpy 求逆与课件一致:', np.allclose(np.linalg.inv(A), inv_expected))
print('验证 A · A⁻¹ = I:\n', np.round(A @ inv_expected, 6))
assert np.allclose(A @ inv_expected, np.eye(3))
print('✅ 伴随矩阵求逆 与课件一致。')

**🔗 Aurora 连接**：协方差矩阵 `Σ` 在 Aurora 的特征白化步骤（`src/aurora/audio/features.py`）中必须可逆；`det(Σ) = 0` 意味着某个频带的能量完全相关，白化矩阵 `Σ⁻¹/²` 无法计算，需要先加正则项 `ε·I`。Mahalanobis 距离 `d² = (x−μ)ᵀ Σ⁻¹ (x−μ)` 直接依赖 `Σ⁻¹`——奇异协方差会让距离计算数值爆炸。

**补充例题对应**：行列式、子式 Minor、代数余子式 Cofactor、伴随 adjoint、逆。

## 🎨 图示：矩阵的列(行列式 = 列张成的有向面积)

In [ ]:
from laviz import style, matrix_4ways
style()
matrix_4ways([[1,1,0],[1,2,1],[0,1,3]]);

In [ ]:
A = np.array([[2.0, 1.0], [0.0, 1.0]])
probes = np.array([[1,0], [0,1], [1,1], [-1,2]], dtype=float)
print('矩阵 A 会怎样移动这些向量？')
for v in probes:
    out = A @ v
    print(f'{v} -> {out} | 长度 {np.linalg.norm(v):.2f} -> {np.linalg.norm(out):.2f}')


## 参数实验：随机矩阵与奇异矩阵对比

构造 `A = np.random.rand(3, 3)` 多次，打印 `np.linalg.det(A)`，确认随机 3×3 矩阵几乎总是可逆的。再把第二行设成第一行的两倍：`A[1] = 2 * A[0]`，重新计算 det，确认它变为 0（或因浮点舍入极接近 0）。同时用 `np.linalg.cond(A)` 对比两种情况：随机矩阵条件数通常在 10–100 之间，奇异矩阵条件数爆到 1e15 以上。

In [ ]:
A = np.array([[2.0, 1.0], [0.0, 1.0]])
probes = np.array([[1,0], [0,1], [1,1], [-1,2]], dtype=float)
print('矩阵 A 会怎样移动这些向量？')
for v in probes:
    out = A @ v
    print(f'{v} -> {out} | 长度 {np.linalg.norm(v):.2f} -> {np.linalg.norm(out):.2f}')


## 本课收束

现在能用 `det_2x2` 和 `inv_2x2` 验证任意 2×2 矩阵是否可逆，用余子式展开处理 3×3 情形。在 Aurora 协方差白化步骤中，先算 det 来决定能否安全调用求逆。下一节的特征分解会用 det=0 来识别奇异矩阵，这是今天结论的直接延伸。

In [ ]:
# 小检查：先看 shape，再做运算
import numpy as np

a = np.array([1.0, 2.0])
b = np.array([3.0, 4.0])
A = np.array([[1.0, 2.0], [0.0, 1.0]])

print('a.shape =', a.shape)
print('b.shape =', b.shape)
print('A.shape =', A.shape)
print('a + b =', a + b)
print('A @ a =', A @ a)
print('a dot b =', float(a @ b))
